# Skip Mechanism Diagnostics

Read-only checks for duplicate registry rows and DAS classification skip behavior.
This notebook does not modify existing project files.

In [ ]:
from pathlib import Path

import pandas as pd

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR

BASE_REGISTRY = Path(REGISTRIES_DIR) / "base_output_registry.csv"
EXTRACTION_REGISTRY = Path(SUBREGISTRIES_DIR) / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY = Path(SUBREGISTRIES_DIR) / "DAS_classification_output_registry.csv"

df_base = pd.read_csv(BASE_REGISTRY)
df_extraction = pd.read_csv(EXTRACTION_REGISTRY)
df_classification = pd.read_csv(CLASSIFICATION_REGISTRY)

print("base:", df_base.shape)
print("extraction:", df_extraction.shape)
print("classification:", df_classification.shape)

In [ ]:
# Exact duplicate rows in each registry, ignoring creation_date in base.
base_subset = [c for c in df_base.columns if c != "creation_date"]

print("base exact duplicates:", df_base.duplicated(subset=base_subset).sum())
print("extraction exact duplicates:", df_extraction.duplicated().sum())
print("classification exact duplicates:", df_classification.duplicated().sum())

dup_base = df_base[df_base.duplicated(subset=base_subset, keep=False)]
dup_base["output_type"].value_counts()

In [ ]:
# Where do extracted-section duplicates in base come from?
base_extracted = df_base[df_base["output_type"] == "extracted section"].copy()
extraction_meta = df_extraction[["output_sha", "section_type", "stage"]].drop_duplicates()
base_extracted = base_extracted.merge(extraction_meta, on="output_sha", how="left")

print(base_extracted.groupby(["stage", "software_version"]).size().sort_index())

preclean_dups = base_extracted[
    base_extracted.duplicated(subset=["output_sha"], keep=False)
    & (base_extracted["stage"] == "pre-cleaning")
]
print("duplicate pre-cleaning extracted-section rows in base:", len(preclean_dups))
preclean_dups[["output_sha", "doc_doi", "software_version", "dependencies"]].head(10)

In [ ]:
# Reproduce the current classify_sections dependency_set logic.
classification_shas = set(df_classification.loc[df_classification["method"] == "LLM", "output_sha"])
dependency_set = set(df_base.loc[df_base["output_sha"].isin(classification_shas), "dependencies"].dropna())

print("dependency_set size:", len(dependency_set))
print("sample:", list(sorted(dependency_set))[:10])

In [ ]:
# Reproduce the current classify_sections candidate selection.
merged = df_extraction.merge(
    df_base[["output_sha", "doc_doi", "software_version"]],
    on="output_sha",
    how="left",
)

cleaned_das = merged[(merged["section_type"] == "DAS") & (merged["stage"] == "cleaned")].copy()

summary_rows = []
for sw in sorted(cleaned_das["software_version"].dropna().unique()):
    candidate_shas = set(cleaned_das.loc[cleaned_das["software_version"] == sw, "output_sha"])
    skipped = sum(sha in dependency_set for sha in candidate_shas)
    summary_rows.append(
        {
            "software_version": sw,
            "candidate_cleaned_DAS": len(candidate_shas),
            "would_skip_now": skipped,
            "would_run_now": len(candidate_shas) - skipped,
        }
    )

pd.DataFrame(summary_rows)

In [ ]:
# Compare classification rows in base vs classification_output_registry.
base_class = df_base[df_base["output_type"] == "classification"].copy()
base_class["present_in_classification_registry"] = base_class["output_sha"].isin(classification_shas)

base_class.groupby(["software_version", "present_in_classification_registry"]).size().unstack(fill_value=0)

In [ ]:
# Exact duplicate classification rows inside base.
base_class_dups = base_class[base_class.duplicated(subset=base_subset, keep=False)]
print("duplicate classification rows in base:", len(base_class_dups))
base_class_dups[["output_sha", "doc_doi", "software_version", "dependencies"]].sort_values(
    ["software_version", "doc_doi", "output_sha"]
).head(20)